# DS2/GSE141383 — dataset-level quality control

This notebook reproduces the QC sequence recorded in the original
GSE141383 notebook while removing unrelated scVI/GSE154795 code and
machine-specific paths.

## Filtering sequence

1. calculate QC metrics from raw integer counts;
2. convert the feature axis from versioned Ensembl identifiers to gene
   symbols while retaining the original Ensembl identifiers in
   `adata.var["ensembl_id_version"]`;
3. retain cells satisfying all of:
   - detected genes > 300;
   - detected genes < 4,500;
   - total counts > 600;
   - total counts < 12,000;
   - mitochondrial fraction < 3%;
4. apply the final total-count ceiling of 9,000;
5. retain genes detected in at least 10 final cells.

Raw counts remain in both `adata.X` and `adata.layers["counts"]`.
No normalization or log transformation is performed.

## Expected output

- 21,391 cells
- 25,835 genes

Input:
`data/interim/DS2_GSE141383/GSE141383_merged.h5ad`

Output:
`data/processed/DS2_GSE141383/GSE141383_postQC.h5ad`

In [ ]:
import json
import os
import sys
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp

print("Python:", sys.version)
print("scanpy:", sc.__version__)
print("pandas:", pd.__version__)

In [ ]:
def find_repository_root(start: Path) -> Path:
    start = start.expanduser().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "README.md").exists() and (candidate / "code").exists():
            return candidate
    return start


PROJECT_ROOT = Path(
    os.environ.get("GLIOMA_PROJECT_ROOT", find_repository_root(Path.cwd()))
).expanduser().resolve()

INPUT_H5AD = Path(
    os.environ.get(
        "GSE141383_MERGED_H5AD",
        PROJECT_ROOT
        / "data"
        / "interim"
        / "DS2_GSE141383"
        / "GSE141383_merged.h5ad",
    )
).expanduser().resolve()

OUTPUT_H5AD = Path(
    os.environ.get(
        "GSE141383_POSTQC_H5AD",
        PROJECT_ROOT
        / "data"
        / "processed"
        / "DS2_GSE141383"
        / "GSE141383_postQC.h5ad",
    )
).expanduser().resolve()

AUDIT_DIR = (
    PROJECT_ROOT / "results" / "qc" / "DS2_GSE141383" / "00b_qc"
)
FIGURE_DIR = AUDIT_DIR / "figures"

OUTPUT_H5AD.parent.mkdir(parents=True, exist_ok=True)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_INPUT_SHAPE = (29_415, 60_725)
EXPECTED_OUTPUT_SHAPE = (21_391, 25_835)
EXPECTED_AFTER_PRIMARY_CELL_FILTER = 21_500

MIN_GENES_EXCLUSIVE = 300
MAX_GENES_EXCLUSIVE = 4_500
MIN_COUNTS_EXCLUSIVE = 600
MAX_COUNTS_PRIMARY_EXCLUSIVE = 12_000
MAX_MT_PERCENT_EXCLUSIVE = 3.0
FINAL_MAX_COUNTS_INCLUSIVE = 9_000
FINAL_MIN_CELLS_PER_GENE = 10

print("Input:", INPUT_H5AD)
print("Output:", OUTPUT_H5AD)

if not INPUT_H5AD.exists():
    raise FileNotFoundError(
        f"Merged input not found: {INPUT_H5AD}\n"
        "Run 00a_ds2_merge_per_sample_h5ad.ipynb first or set "
        "GSE141383_MERGED_H5AD."
    )

In [ ]:
def matrix_values(matrix):
    return matrix.data if sp.issparse(matrix) else np.asarray(matrix).ravel()


def assert_raw_counts(matrix, label: str):
    values = matrix_values(matrix)
    if values.size == 0:
        raise ValueError(f"{label} contains no nonzero count values.")
    if not np.isfinite(values).all():
        raise ValueError(f"{label} contains non-finite values.")
    if np.any(values < 0):
        raise ValueError(f"{label} contains negative values.")
    if not np.equal(values, np.floor(values)).all():
        raise ValueError(f"{label} is not an integer raw-count matrix.")


def save_current_figure(path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.close("all")

In [ ]:
adata = sc.read_h5ad(INPUT_H5AD)

if adata.shape != EXPECTED_INPUT_SHAPE:
    raise AssertionError(
        f"Expected input shape {EXPECTED_INPUT_SHAPE}, observed {adata.shape}"
    )
if "sample" not in adata.obs:
    raise KeyError("adata.obs['sample'] is required.")
if "gene_symbols" not in adata.var:
    if "gene_symbol" in adata.var:
        adata.var["gene_symbols"] = adata.var["gene_symbol"]
    else:
        raise KeyError(
            "adata.var must contain 'gene_symbols' or 'gene_symbol'."
        )
if not adata.obs_names.is_unique:
    raise AssertionError("Cell identifiers are not unique.")
if not adata.var_names.is_unique:
    raise AssertionError("Input Ensembl feature identifiers are not unique.")

assert_raw_counts(adata.X, "adata.X")

if sp.issparse(adata.X):
    adata.X = adata.X.tocsr()
adata.layers["counts"] = adata.X.copy()
assert_raw_counts(adata.layers["counts"], 'adata.layers["counts"]')

print(adata)

In [ ]:
gene_symbols = adata.var["gene_symbols"].fillna("").astype(str)
gene_symbols_upper = gene_symbols.str.upper()

adata.var["mt"] = gene_symbols_upper.str.startswith("MT-")
adata.var["ribo"] = gene_symbols_upper.str.startswith(("RPS", "RPL"))
adata.var["hb"] = gene_symbols_upper.str.match(r"^HB[ABDEGQMZ]")

sc.pp.calculate_qc_metrics(
    adata,
    qc_vars=["mt", "ribo", "hb"],
    percent_top=[50, 100, 200, 500],
    log1p=True,
    inplace=True,
)

required_columns = [
    "n_genes_by_counts",
    "total_counts",
    "pct_counts_mt",
    "pct_counts_ribo",
    "pct_counts_hb",
]
missing = [column for column in required_columns if column not in adata.obs]
if missing:
    raise RuntimeError(f"QC metric calculation failed: missing {missing}")

print("Mitochondrial genes:", int(adata.var["mt"].sum()))
print(adata.obs[required_columns].describe())

In [ ]:
# Retain the original versioned Ensembl identifiers before switching the
# feature axis to gene symbols, as in the original analysis.
adata.var["ensembl_id_version"] = adata.var_names.astype(str)

symbol_index = adata.var["gene_symbols"].fillna("").astype(str)
invalid_symbol = symbol_index.str.strip().isin({"", "nan", "None", "NA"})
if invalid_symbol.any():
    symbol_index = symbol_index.copy()
    symbol_index.loc[invalid_symbol] = adata.var.loc[
        invalid_symbol, "ensembl_id_version"
    ].astype(str)

adata.var_names = pd.Index(symbol_index, name="gene_symbol")
adata.var_names_make_unique()

if not adata.var_names.is_unique:
    raise AssertionError(
        "Gene-symbol feature identifiers are not unique after repair."
    )

print("Example feature names:", adata.var_names[:10].tolist())

In [ ]:
primary_rule_masks = pd.DataFrame(index=adata.obs_names)
primary_rule_masks["pass_min_genes"] = (
    adata.obs["n_genes_by_counts"] > MIN_GENES_EXCLUSIVE
)
primary_rule_masks["pass_max_genes"] = (
    adata.obs["n_genes_by_counts"] < MAX_GENES_EXCLUSIVE
)
primary_rule_masks["pass_min_counts"] = (
    adata.obs["total_counts"] > MIN_COUNTS_EXCLUSIVE
)
primary_rule_masks["pass_primary_max_counts"] = (
    adata.obs["total_counts"] < MAX_COUNTS_PRIMARY_EXCLUSIVE
)
primary_rule_masks["pass_max_mt"] = (
    adata.obs["pct_counts_mt"] < MAX_MT_PERCENT_EXCLUSIVE
)
primary_rule_masks["pass_primary_QC"] = primary_rule_masks.all(axis=1)

primary_rule_summary = pd.DataFrame(
    {
        "rule": primary_rule_masks.columns,
        "cells_passing": [
            int(primary_rule_masks[column].sum())
            for column in primary_rule_masks
        ],
        "cells_failing": [
            int((~primary_rule_masks[column]).sum())
            for column in primary_rule_masks
        ],
    }
)
primary_rule_summary.to_csv(
    AUDIT_DIR / "GSE141383_primary_QC_rule_summary.tsv",
    sep="\t",
    index=False,
)

adata_primary = adata[
    primary_rule_masks["pass_primary_QC"].to_numpy()
].copy()

if adata_primary.n_obs != EXPECTED_AFTER_PRIMARY_CELL_FILTER:
    raise AssertionError(
        "Primary DS2 cell filtering did not reproduce the original count. "
        f"Expected {EXPECTED_AFTER_PRIMARY_CELL_FILTER}, observed "
        f"{adata_primary.n_obs}."
    )

print("After primary cell filtering:", adata_primary.shape)
primary_rule_summary

In [ ]:
before_sample_counts = (
    adata.obs.groupby("sample", observed=True)
    .size()
    .rename("cells_before_QC")
)

final_cell_mask = (
    adata_primary.obs["total_counts"] <= FINAL_MAX_COUNTS_INCLUSIVE
)
adata_qc = adata_primary[final_cell_mask.to_numpy()].copy()

sc.pp.filter_genes(
    adata_qc,
    min_cells=FINAL_MIN_CELLS_PER_GENE,
)

after_sample_counts = (
    adata_qc.obs.groupby("sample", observed=True)
    .size()
    .rename("cells_after_QC")
)

sample_retention = pd.concat(
    [before_sample_counts, after_sample_counts],
    axis=1,
).fillna(0)
sample_retention["cells_removed"] = (
    sample_retention["cells_before_QC"]
    - sample_retention["cells_after_QC"]
)
sample_retention["retention_fraction"] = (
    sample_retention["cells_after_QC"]
    / sample_retention["cells_before_QC"]
)
sample_retention.to_csv(
    AUDIT_DIR / "GSE141383_QC_retention_by_sample.tsv",
    sep="\t",
)

if adata_qc.shape != EXPECTED_OUTPUT_SHAPE:
    raise AssertionError(
        "DS2 QC did not reproduce the recorded output shape. "
        f"Expected {EXPECTED_OUTPUT_SHAPE}, observed {adata_qc.shape}. "
        "Confirm that the same 29,415-cell merged object and compatible "
        "Scanpy/AnnData versions are being used."
    )

assert_raw_counts(adata_qc.X, "post-QC adata.X")
assert_raw_counts(
    adata_qc.layers["counts"],
    'post-QC adata.layers["counts"]',
)

print("Final post-QC object:", adata_qc)
sample_retention

In [ ]:
sc.pl.violin(
    adata,
    ["n_genes_by_counts", "total_counts", "pct_counts_mt"],
    jitter=0.4,
    multi_panel=True,
    show=False,
)
save_current_figure(FIGURE_DIR / "GSE141383_QC_metrics_before_filtering.png")

sc.pl.violin(
    adata_qc,
    ["n_genes_by_counts", "total_counts", "pct_counts_mt"],
    jitter=0.4,
    multi_panel=True,
    show=False,
)
save_current_figure(FIGURE_DIR / "GSE141383_QC_metrics_after_filtering.png")

sc.pl.scatter(
    adata,
    x="total_counts",
    y="n_genes_by_counts",
    color="pct_counts_mt",
    show=False,
)
save_current_figure(FIGURE_DIR / "GSE141383_scatter_before_filtering.png")

sc.pl.scatter(
    adata_qc,
    x="total_counts",
    y="n_genes_by_counts",
    color="pct_counts_mt",
    show=False,
)
save_current_figure(FIGURE_DIR / "GSE141383_scatter_after_filtering.png")

In [ ]:
qc_summary = {
    "dataset_id": "DS2_GSE141383",
    "geo_accession": "GSE141383",
    "input_cells": int(adata.n_obs),
    "input_features": int(adata.n_vars),
    "post_primary_filter_cells": int(adata_primary.n_obs),
    "post_QC_cells": int(adata_qc.n_obs),
    "post_QC_genes": int(adata_qc.n_vars),
    "retention_fraction": float(adata_qc.n_obs / adata.n_obs),
    "min_genes_exclusive": MIN_GENES_EXCLUSIVE,
    "max_genes_exclusive": MAX_GENES_EXCLUSIVE,
    "min_counts_exclusive": MIN_COUNTS_EXCLUSIVE,
    "primary_max_counts_exclusive": MAX_COUNTS_PRIMARY_EXCLUSIVE,
    "max_mt_percent_exclusive": MAX_MT_PERCENT_EXCLUSIVE,
    "final_max_counts_inclusive": FINAL_MAX_COUNTS_INCLUSIVE,
    "final_min_cells_per_gene": FINAL_MIN_CELLS_PER_GENE,
    "feature_axis": "gene symbols made unique",
    "original_feature_id_column": "ensembl_id_version",
    "raw_counts_in_X": True,
    "raw_counts_layer": "counts",
    "normalization_applied": False,
    "timestamp": datetime.now().isoformat(timespec="seconds"),
    "output_h5ad": str(OUTPUT_H5AD),
}

adata_qc.uns["dataset_level_QC"] = qc_summary

with open(
    AUDIT_DIR / "GSE141383_QC_summary.json",
    "w",
    encoding="utf-8",
) as handle:
    json.dump(qc_summary, handle, indent=2)

print(qc_summary)

In [ ]:
temporary_output = OUTPUT_H5AD.with_suffix(".tmp.h5ad")
if temporary_output.exists():
    temporary_output.unlink()

adata_qc.write_h5ad(temporary_output, compression="gzip")
temporary_output.replace(OUTPUT_H5AD)

with sc.read_h5ad(OUTPUT_H5AD, backed="r") as check:
    if check.shape != EXPECTED_OUTPUT_SHAPE:
        raise AssertionError(
            f"Saved-file validation failed: observed {check.shape}"
        )
    if "counts" not in check.layers:
        raise AssertionError("Saved output is missing layers['counts'].")
    if "ensembl_id_version" not in check.var:
        raise AssertionError(
            "Saved output is missing var['ensembl_id_version']."
        )

print("Saved and validated:", OUTPUT_H5AD)